# Stage 4: 多方法嵌入（等权比较）+ 显式遍历

运行 PCA（基线）、Harmony、scVI（从头训练），以及可选
scANVI（存在标注参考时）和 cellxgene_census 预训练 scVI。

所有嵌入方法**等权并列**——无优先级预设。选择依据：
1. **可视化检查（首要）**——按 sample_id、batch 和 cell_type（如有）着色的
   UMAP 图。PI 目视判断各方法在批次整合与生物学信号保留之间的平衡。
2. **整合指标（佐证）**——显式 for 循环遍历所有嵌入，对每个调
   `integration_metrics(adata)` 生成对比表。无回调、无 sweep() 抽象。

**本 notebook 产出**：
- `obsm["X_pca"]`——PCA（基线，仅 HVG）
- `obsm["X_pca_harmony"]`——Harmony 批次校正 PCA
- `obsm["X_scVI"]`——scVI 潜变量（从头训练）
- `obsm["X_scANVI"]`——scANVI 潜变量（仅当存在标注参考时）
- 各嵌入的 UMAP 图：按 sample_id、batch、cell_type 着色
- `results/figures/sweep_stage4/` 中显式遍历产出对比表（含整合指标）
- `adata.uns` 运行元数据（`harmony_v1`、`scvi_v1` 等）
- Stage 4 checkpoint h5ad，供 stage 5 聚类使用

**增加新嵌入方法**（扩展模式）：写 `adata.obsm["X_{method}"]`，
并将 `"X_{method}"` 加入下方 `use_reps` 列表——一个 cell，零框架改动。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：Stage 3（标准化 + HVG），读 `stage3_normalized_v*.h5ad`
- **下游**：Stage 5（多分辨率 Leiden 聚类），产出 `stage4_embedded_v*.h5ad`

### 为什么要迭代回跑？
嵌入质量直接影响下游分群和注释的准确性。如果在 stage 5（Leiden 分群不合理）、
stage 6（注释时发现嵌入没有分离应区分的细胞类型）发现问题，可能需要：
- 换用不同的嵌入方法（增减 `EMBEDDING_METHODS` 列表）
- 调整 PCA 维度数（`N_PCS`）
- 增加 scVI 训练轮数（`SCVI_MAX_EPOCHS`——默认 20 是快速验证用，生产建议 200-400）
- 换用 stage 3 的另一个版本（不同 HVG 数量）

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `nancang_stage3_normalized_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`
   （例如 `nancang_stage4_embedded_v2.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改嵌入方法或训练参数）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为嵌入质量可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：stage 5 的 `UPSTREAM_PATH` 指向你决定采用的 stage 4 版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"stage4_embedded"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"stage 4 有哪些版本？哪些依赖 stage3_v1？"，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH       — stage 3 产出文件路径。
#                        如需回跑：指向要复用的上游版本（如 stage3_normalized_v2.h5ad）。
# OUTPUT_PATH         — 本 stage 产出 checkpoint 路径。
#                        版本号 _v1 与 adata.uns["version"] 保持一致。
#                        如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# N_PCS               — PCA 维度数（Harmony 也使用此值）。
# BATCH_KEY           — obs 中批次变量列名（Harmony、scVI 使用）。
#                        Nancang fixture 用 "sample_id"（6 个 GSM 样本）。
# EMBEDDING_METHODS   — 要计算的嵌入方法列表。默认 PCA + Harmony + scVI。
#                        需增减方法时修改此列表即可。
# SCVI_MAX_EPOCHS     — scVI 训练轮数。快速验证管线用 20；
#                        生产质量嵌入建议 200-400。
# SCVI_N_LAYERS       — scVI 编码器/解码器深度（标准值 2）。
# SCVI_N_LATENT       — scVI 潜空间维度。
# RANDOM_SEED         — 固定随机种子，保证可复现。

UPSTREAM_PATH = "results/nancang_stage3_normalized_v1.h5ad"
OUTPUT_PATH   = "results/nancang_stage4_embedded_v1.h5ad"

N_PCS              = 30
BATCH_KEY          = "sample_id"   # 每个数据集必须核对 BATCH_KEY 实际列名；Nancang fixture 用 "sample_id"
EMBEDDING_METHODS  = ["pca", "harmony", "scvi"]   # 始终跑 baseline + Harmony + scVI
# EMBEDDING_METHODS.append("scanvi")               # 有标注 atlas 时取消注释
# EMBEDDING_METHODS.append("cellxgene_census")     # Census 有此组织模型时取消注释
SCVI_MAX_EPOCHS    = 20    # 快速验证：低 epochs 加速管线调试
                            # 生产质量：增加到 200-400
SCVI_N_LAYERS      = 2     # scVI 编码器/解码器深度（2 为标准值）
SCVI_N_LATENT      = 10    # scVI 潜空间维度
RANDOM_SEED        = 42

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/figures/sweep_stage4", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

In [ ]:
# 导入（scanpy 原生 API + 框架函数仅在真正需要时使用）。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import matplotlib.pyplot as plt
import datetime
import warnings

# 直接从 scorers 模块导入指标函数——无回调抽象，在 for 循环中直接调用
from scrna_integration.scorers import integration_metrics

# 抑制 scvi-tools PyTorch Lightning 弃用警告
warnings.filterwarnings("ignore", message=".*Lightning.*")
warnings.filterwarnings("ignore", message=".*The number of training batches.*")

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

print("加载上游:", UPSTREAM_PATH)
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"layers: {list(adata.layers.keys())}")
print(f"obsm keys: {list(adata.obsm.keys())}")

## PCA（基线）

在高可变基因（HVG）上做主成分分析。这是最简单的嵌入方法——
无批次校正、无深度模型。作为所有整合方法的基线对照。

In [ ]:
# 在高可变基因（HVG）上做主成分分析。
if "pca" in EMBEDDING_METHODS:
    print(f"\n===== PCA: n_comps={N_PCS} =====")
    sc.tl.pca(adata, n_comps=N_PCS, use_highly_variable=True,
              svd_solver="arpack", random_state=RANDOM_SEED)
    print(f"obsm['X_pca'] shape: {adata.obsm['X_pca'].shape}")
    var_explained = adata.uns['pca']['variance_ratio'].sum() * 100
    print(f"Cumulative variance explained (top {N_PCS} PCs): {var_explained:.1f}%")
else:
    print("PCA skipped (not in EMBEDDING_METHODS)")


## Harmony

Harmony 通过迭代校正 PCA 嵌入来整合批次。速度快、确定性算法，
在单细胞分析管线中广泛使用。与 PCA 共用相同 `N_PCS` 参数，
batch key 来自 PARAMS 单元格。

In [ ]:
# Harmony 在 PCA 嵌入上做批次整合。
# 注意：直接使用 harmonypy（而非 sc.external.pp.harmony_integrate），
# 因为 harmonypy >= 2.0.0 把 Z_corr 方向从 (dims, cells) 改为
# (cells, dims)；scanpy wrapper 仍假设旧布局做 .T 导致 shape 不匹配。
# 故本 cell 直接调用 harmonypy 避免 scanpy wrapper 的兼容问题。
if "harmony" in EMBEDDING_METHODS:
    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中，可用列: {list(adata.obs.columns)}；跳过 Harmony 去批次")
    else:
        print(f"\n===== Harmony: batch_key='{BATCH_KEY}' =====")
        import harmonypy

        ho = harmonypy.run_harmony(
            adata.obsm["X_pca"],
            adata.obs,
            BATCH_KEY,
            max_iter_harmony=10,
        )
        # ho.Z_corr 在 harmonypy >= 2.0.0 中已经是 (n_cells, n_pcs) 方向。
        assert ho.Z_corr.shape[1] == N_PCS, f"harmonypy Z_corr shape mismatch: {ho.Z_corr.shape}; check harmonypy version API"
        adata.obsm["X_pca_harmony"] = ho.Z_corr
        print(f"obsm['X_pca_harmony'] shape: {adata.obsm['X_pca_harmony'].shape}")
else:
    print("Harmony skipped (not in EMBEDDING_METHODS)")


## scVI（从头训练）

scVI（单细胞变分推断）学习一个显式将批次效应建模为干扰变量的潜表示。
需要原始 counts——使用 `adata.layers['counts']`（stage 3 保留的原始计数）。

**快速验证说明**：上方 `SCVI_MAX_EPOCHS` 设得较低以加速管线调试。
生产质量嵌入建议增加到 200-400 epochs。监控训练损失曲线确认收敛。

**为什么至少需要 20 epochs？** scVI 的变分推断需要足够迭代才能收敛到
有意义的潜空间。太少的 epochs 会导致嵌入质量差、下游 UMAP 和聚类
无法反映真实的生物学结构。200-400 epochs 是 scVI 论文和社区的推荐值。

scVI 设置步骤：
1. `setup_anndata`——声明哪些 layer/列存放 counts、batch 等
2. `SCVI(adata)`——实例化模型
3. `.train()`——训练（含早停）
4. `.get_latent_representation()`——提取嵌入到 `obsm["X_scVI"]`

In [ ]:
# scVI：setup_anndata + 训练 + 提取潜表示。
if "scvi" in EMBEDDING_METHODS:
    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中，可用列: {list(adata.obs.columns)}；跳过 scVI")
    else:
        print(f"\n===== scVI: max_epochs={SCVI_MAX_EPOCHS}, n_latent={SCVI_N_LATENT} =====")
        import scvi

        scvi.model.SCVI.setup_anndata(
            adata,
            layer="counts",
            batch_key=BATCH_KEY,
        )

        model = scvi.model.SCVI(
            adata,
            n_layers=SCVI_N_LAYERS,
            n_latent=SCVI_N_LATENT,
            gene_likelihood="zinb",
            use_layer_norm="both",
            use_batch_norm="none",
        )

        print(f"Training scVI (max_epochs={SCVI_MAX_EPOCHS})...")
        model.train(
            max_epochs=SCVI_MAX_EPOCHS,
            early_stopping=True,
            early_stopping_patience=5,
            plan_kwargs={"lr": 1e-3},
            check_val_every_n_epoch=1,
        )

        adata.obsm["X_scVI"] = model.get_latent_representation()
        print(f"obsm['X_scVI'] shape: {adata.obsm['X_scVI'].shape}")
        print("scVI complete.")
else:
    print("scVI skipped (not in EMBEDDING_METHODS)")


## (Optional) cellxgene_census pretrained scVI

Use a CZ CELLxGENE Census pretrained scVI model to embed this dataset.
The pretrained model maps new cells into a latent space learned from
millions of cells across the CELLxGENE corpus -- no local training needed.

**PREREQUISITE**: a Census-hosted scVI model must exist for this tissue
(gastric mucosa in the GCPL case). Check availability at cellxgene.cziscience.com.

When enabled, the cell below:
1. Downloads the pretrained model from Census.
2. Aligns gene symbols via mygene.
3. Calls `prepare_query_anndata` and `load_query_data`.
4. Extracts latent representation to `obsm["X_scVI_census"]`.

The cell is **commented out** by default because it adds Census as a
heavy dependency and only works when a good tissue model exists.

In [ ]:
# # === cellxgene_census 预训练 scVI（已注释） ===
# # 前置条件：Census 存在该组织的预训练 scVI 模型。
# # 启用此 cell：取消下方注释，然后在 PARAMS 中添加
# # "cellxgene_census" 到 EMBEDDING_METHODS。
#
# # import cellxgene_census
# # census = cellxgene_census.open_soma(census_version="latest")
# # model = cellxgene_census.download_source_h5ad(
# #     ORGANISM, layer="scvi", tissue=TISSUE,
# # )
# # # 基因对齐（参见 legacy-GCPL/04_dimensionality_reduction.ipynb）
# # adata.obsm["X_scVI_census"] = model.get_latent_representation(adata)
# # print("cellxgene_census scVI complete.")
#
# print("cellxgene_census pretrained scVI cell is commented out. "
#       "Uncomment when Census has a model for this tissue.")

## (Optional) scANVI label transfer embedding

scANVI (single-cell ANnotation Variational Inference) extends scVI with
a cell-type classifier head. It produces a latent space that is both
batch-corrected AND cell-type-aware -- useful when a labelled reference
exists and cell types are expected to be conserved across batches.

**PREREQUISITE**: `adata.obs` must contain a usable cell-type label column.
This cell auto-detects common label column names:
`cell_type_original_*`, `Celltypes_global`, `cell_type`, `label`.
If none found, scANVI is gracefully skipped with a log message.

In [ ]:
# scANVI：scVI 的半监督变体，利用细胞类型标签。
# 无可用标签时优雅跳过并给出说明。
if "scanvi" in EMBEDDING_METHODS:
    print("\n===== scANVI: checking for cell-type label column =====")

    # 自动检测标签列：优先原始作者标注，其次常见列名。
    label_col = None
    for pattern in ["cell_type_original_", "Celltypes_global", "cell_type", "label",
                    "Detailed_Cell_Type", "Global_cluster_selected"]:
        for col in adata.obs.columns:
            if pattern in col:
                label_col = col
                break
        if label_col is not None:
            break

    if label_col is None:
        print("scANVI SKIPPED: no cell-type label column found in adata.obs.")
        print("  Available obs columns:", list(adata.obs.columns)[:10], "...")
        print("  To enable scANVI: add a label column to adata.obs before running.")
    else:
        n_labels = adata.obs[label_col].nunique()
        n_nan = adata.obs[label_col].isna().sum()
        print(f"Found label column: '{label_col}' ({n_labels} unique, {n_nan} NaN)")

        if n_labels < 2:
            print(f"scANVI SKIPPED: '{label_col}' has < 2 unique non-NaN values.")
        else:
            import scvi
            from scvi.model import SCANVI

            scvi.model.SCVI.setup_anndata(
                adata, layer="counts", batch_key=BATCH_KEY,
                labels_key=label_col,
            )
            scvi_model = scvi.model.SCVI(
                adata, n_layers=SCVI_N_LAYERS, n_latent=SCVI_N_LATENT,
                gene_likelihood="zinb", use_layer_norm="both", use_batch_norm="none",
            )
            scvi_model.train(max_epochs=SCVI_MAX_EPOCHS, early_stopping=True,
                             early_stopping_patience=5, plan_kwargs={"lr": 1e-3})

            scanvi_model = SCANVI.from_scvi_model(
                scvi_model,
                unlabeled_category="Unknown",
                labels_key=label_col,
            )
            n_ep = max(5, SCVI_MAX_EPOCHS // 2)
            print(f"Training scANVI (max_epochs={n_ep})...")
            scanvi_model.train(
                max_epochs=n_ep,
                early_stopping=True,
                early_stopping_patience=3,
            )

            adata.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
            print(f"obsm['X_scANVI'] shape: {adata.obsm['X_scANVI'].shape}")
            print("scANVI complete.")
else:
    print("scANVI skipped (not in EMBEDDING_METHODS)")

## 可视化对比（首要决策依据）

对每个已存在的嵌入计算邻居图 + UMAP，并以三种着色方式绘图：
1. **sample_id**——不同样本是否充分混合？（批次整合质量）
2. **batch**——方法是否正确校正了声明的批次变量？
3. **cell_type**（如有）——已知细胞类型是否被分离？

PI **目视检查这些图**来判断哪个嵌入在批次整合和生物学信号保留之间
达到最佳平衡。下方指标表提供定量佐证——但可视化是首要决策机制。

In [ ]:
# 按嵌入逐个计算：邻居图 → UMAP → 三种着色方式的可视化。
print("\n===== Visual comparison: UMAP per embedding =====")

# 收集 obsm 中所有已存在的嵌入。
existing_embeddings = [k for k in adata.obsm.keys()
                       if k.startswith("X_")
                       and adata.obsm[k].shape[1] >= 2]
print(f"Existing embedding keys: {existing_embeddings}")

# 确定可用的着色列。
colour_columns = ["sample_id"]
if BATCH_KEY in adata.obs.columns and BATCH_KEY != "sample_id":
    colour_columns.append(BATCH_KEY)
for ct_cand in ["Celltypes_global", "cell_type", "cell_type_original"]:
    if ct_cand in adata.obs.columns:
        colour_columns.append(ct_cand)
        break
print(f"Colour columns: {colour_columns}")

for embed_key in existing_embeddings:
    print(f"\n--- {embed_key} ---")

    # 在嵌入空间中计算邻居图 + UMAP。
    sc.pp.neighbors(adata, use_rep=embed_key, n_pcs=min(N_PCS, adata.obsm[embed_key].shape[1]),
                     random_state=RANDOM_SEED)
    sc.tl.umap(adata, random_state=RANDOM_SEED)

    # 将 UMAP 坐标保存到专属 key，便于后续追溯。
    umap_key = f"X_umap_{embed_key.lstrip('X_')}"
    adata.obsm[umap_key] = adata.obsm["X_umap"].copy()
    print(f"  UMAP saved to obsm['{umap_key}']")

    # 按每种着色方式绘图。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            fig_name = f"stage4_umap_{embed_key}_{colour}.png"
            sc.pl.umap(
                adata, color=colour,
                title=f"{embed_key} - coloured by {colour}",
                frameon=False,
                save=f"_{embed_key}_{colour}.png",
            )
            # 从 scanpy 默认 figures/ 目录移至 results/figures/。
            src = f"figures/umap_{embed_key}_{colour}.png"
            dst = f"results/figures/{fig_name}"
            if os.path.exists(src):
                os.rename(src, dst)
                print(f"  Saved {dst}")
            plt.close("all")
        else:
            print(f"  (colour '{colour}' not in obs - skipped)")

## 显式遍历嵌入 + 整合指标（佐证指标）

直接遍历所有已有嵌入，对每个嵌入：
1. 拷贝 adata 避免相互干扰
2. 在该嵌入空间计算邻居图 + UMAP
3. 直接调用 `integration_metrics(adata_copy, batch_key=BATCH_KEY, embed_key=rep)` 获取指标
4. 收集结果到 DataFrame

**没有回调、没有 `sweep()`**——学生打开 notebook 能逐行看懂每一步。

指标（数据允许时计算）：
- **silhouette_batch**——越低 = 批次混合越好
- **silhouette_celltype**——越高 = 生物学信号保留越好
- **scib_available**——1.0 表示 scib-metrics 已安装，0.0 表示未安装

对比表写入 `results/figures/sweep_stage4/sweep_report.md`。
PI 对照 UMAP 图和指标表决定选用哪个嵌入。

In [ ]:
# 显式 for 循环：遍历各嵌入，直接计算 neighbors + UMAP + 整合指标。
# 每一步都是标准 scanpy 操作——学生可以逐行阅读和理解，无需学习回调模型。
print("\n===== 显式遍历嵌入 + 整合指标 =====\n")

import pandas as pd

# 只遍历已计算且存在于 obsm 中的嵌入
use_reps = [k for k in ["X_pca", "X_pca_harmony", "X_scVI", "X_scANVI"]
            if k in adata.obsm]
if not use_reps:
    print("警告: 未在 obsm 中找到任何嵌入。跳过。")
else:
    print(f"遍历 {len(use_reps)} 个嵌入: {use_reps}\n")

    results = []
    for rep in use_reps:
        print(f"--- {rep} ---")

        # 拷贝 AnnData 避免不同嵌入之间干扰（邻居图、UMAP 坐标不共用）
        adata_copy = adata.copy()

        # 在嵌入空间计算邻居图 + UMAP
        n_dim = min(N_PCS, adata_copy.obsm[rep].shape[1])
        sc.pp.neighbors(adata_copy, use_rep=rep, n_pcs=n_dim,
                        random_state=RANDOM_SEED)
        sc.tl.umap(adata_copy, random_state=RANDOM_SEED)

        # 直接调用整合指标函数——显式传入 embed_key=rep 避免 auto-detect 误选其他嵌入
        m = integration_metrics(adata_copy, batch_key=BATCH_KEY, embed_key=rep)
        results.append({"use_rep": rep, **m})

        # 输出当前嵌入的指标摘要
        metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in m.items()
                                if isinstance(v, float) and not np.isnan(v))
        print(f"  指标: {metrics_str}")

    # 收集为 DataFrame 对比表
    sweep_df = pd.DataFrame(results)
    os.makedirs("results/figures/sweep_stage4", exist_ok=True)

    # 写 Markdown 报告（无依赖，纯手写表格）
    lines = ["# Stage 4 嵌入对比报告\n",
             f"**{len(use_reps)} 个嵌入** 已评估。\n",
             "## 指标表\n"]
    lines.append("| " + " | ".join(sweep_df.columns) + " |")
    lines.append("|" + "|".join(" --- " for _ in sweep_df.columns) + "|")
    for _, row in sweep_df.iterrows():
        vals = []
        for col in sweep_df.columns:
            v = row[col]
            if isinstance(v, float):
                vals.append(f"{v:.4f}" if not np.isnan(v) else "N/A")
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    with open("results/figures/sweep_stage4/sweep_report.md", "w") as f:
        f.write("\n".join(lines) + "\n")

    # 展示对比表
    print("\n整合指标对比表:")
    try:
        from IPython.display import display as ipy_display
        ipy_display(sweep_df)
    except ImportError:
        print(sweep_df)

    print("\n对比报告: results/figures/sweep_stage4/sweep_report.md")
    adata.uns["stage4_sweep_v1"] = {
        "embeddings_swept": use_reps,
        "scorer": "integration_metrics",
        "report_dir": "results/figures/sweep_stage4",
        "timestamp": datetime.datetime.now().isoformat(),
    }

## 运行元数据——plain `adata.uns` 写入（SPEC 规范）

版本化键（`harmony_v1`、`scvi_v1` 等）记录每个方法以什么参数运行。
PI 用于追踪溯源和重新运行。

In [ ]:
# 记录每个方法的运行元数据（版本化键，支持重跑时共存）。
print("\n===== Run metadata =====")

if "X_pca" in adata.obsm:
    adata.uns["pca_v1"] = {
        "method": "pca",
        "n_comps": N_PCS,
        "use_hvg": True,
        "svd_solver": "arpack",
        "obsm_key": "X_pca",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_pca_harmony" in adata.obsm:
    adata.uns["harmony_v1"] = {
        "method": "harmony",
        "batch_key": BATCH_KEY,
        "n_pcs": N_PCS,
        "obsm_key": "X_pca_harmony",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_scVI" in adata.obsm:
    adata.uns["scvi_v1"] = {
        "method": "scVI",
        "batch_key": BATCH_KEY,
        "n_latent": SCVI_N_LATENT,
        "n_layers": SCVI_N_LAYERS,
        "max_epochs": SCVI_MAX_EPOCHS,
        "gene_likelihood": "zinb",
        "obsm_key": "X_scVI",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_scANVI" in adata.obsm:
    adata.uns["scanvi_v1"] = {
        "method": "scANVI",
        "batch_key": BATCH_KEY,
        "n_latent": SCVI_N_LATENT,
        "obsm_key": "X_scANVI",
        "timestamp": datetime.datetime.now().isoformat(),
    }

# 统一追踪字段——stage + version（与上游字段合并，保持 stage3-7 命名一致）
adata.uns["stage"] = "stage4_embedded"     # 本 stage 标识
adata.uns["version"] = "v1"                  # 与 OUTPUT_PATH 版本号一致

# 上游溯源信息
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"

# 已运行的方法汇总。
active_embeddings = [k for k in adata.obsm.keys()
                     if k.startswith("X_pca") or k.startswith("X_sc")]
print(f"Embeddings produced: {active_embeddings}")
print(f"status: {adata.uns['status']}")
for k in sorted(adata.uns.keys()):
    if k.endswith("_v1"):
        print(f"  {k}: {list(adata.uns[k].keys())}")

## Float32 转换——Memory Discipline #5

scVI/scANVI 输出默认 float64。转为 float32 内存减半，
对单细胞数据的有效精度无实质影响。
**为什么 float32 够用？** 单细胞 counts 和嵌入携带的信息精度
远低于 float64 的 15 位有效数字。float64 只是白白浪费内存。

In [ ]:
# 将所有 obsm 潜变量矩阵转为 float32（内存纪律 #5）。
print("\n===== Float32 cast =====")
for key in list(adata.obsm.keys()):
    if adata.obsm[key].dtype != np.float32:
        adata.obsm[key] = adata.obsm[key].astype(np.float32)
        print(f"  obsm['{key}'] cast to float32")
print("All obsm dtypes:")
for key in adata.obsm:
    print(f"  obsm['{key}']: shape={adata.obsm[key].shape}, dtype={adata.obsm[key].dtype}")

In [ ]:
# 内存纪律自检——写入前一次断言（SPEC Memory Discipline 规范）。
# 守卫最高影响的内存退化：adata.X 变 dense 或丢失 float32。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X invariants violated: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("Memory self-check passed: X is sparse CSR float32.")

In [ ]:
# 将本 stage 产出写出到磁盘（内存纪律 #4：lzf 压缩）。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"Wrote {OUTPUT_PATH}")

import os
assert os.path.exists(OUTPUT_PATH), f"Output NOT found: {OUTPUT_PATH}"
print(f"Verified: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 跨 stage 边界释放内存（内存纪律 #3）。
# 不释放的话 Jupyter kernel 会一直持有上一 stage 的 AnnData，
# 后续 stage 在同一 kernel 中累积导致 OOM。
del adata
import gc
gc.collect()
print("Memory released.")